# Python Data Engineering Interview Notebook

This notebook contains medium and hard Python and pandas interview exercises for data engineering practice.

How to use it:
- Read the prompt in each section.
- Implement your solution in the code cell below it.
- Run the validation cells where provided.
- Prefer readable, production-style code over clever one-liners.

In [16]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict, deque
from datetime import datetime, timedelta

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## Shared Sample Data

In [17]:
orders = pd.DataFrame([
    {"order_id": 1, "customer_id": "C1", "order_ts": "2026-01-01 10:00:00", "amount": 120.0, "status": "paid"},
    {"order_id": 2, "customer_id": "C2", "order_ts": "2026-01-01 10:05:00", "amount": 80.0, "status": "paid"},
    {"order_id": 3, "customer_id": "C1", "order_ts": "2026-01-01 11:00:00", "amount": 50.0, "status": "refunded"},
    {"order_id": 4, "customer_id": "C3", "order_ts": "2026-01-02 09:15:00", "amount": 200.0, "status": "paid"},
    {"order_id": 5, "customer_id": "C2", "order_ts": "2026-01-02 09:30:00", "amount": 60.0, "status": "paid"},
    {"order_id": 6, "customer_id": "C1", "order_ts": "2026-01-03 12:00:00", "amount": 150.0, "status": "paid"},
])
orders["order_ts"] = pd.to_datetime(orders["order_ts"])

order_items = pd.DataFrame([
    {"order_id": 1, "sku": "A", "qty": 2, "unit_price": 40.0},
    {"order_id": 1, "sku": "B", "qty": 1, "unit_price": 40.0},
    {"order_id": 2, "sku": "B", "qty": 2, "unit_price": 40.0},
    {"order_id": 3, "sku": "C", "qty": 1, "unit_price": 50.0},
    {"order_id": 4, "sku": "A", "qty": 1, "unit_price": 40.0},
    {"order_id": 4, "sku": "D", "qty": 4, "unit_price": 40.0},
    {"order_id": 5, "sku": "C", "qty": 1, "unit_price": 60.0},
    {"order_id": 6, "sku": "A", "qty": 3, "unit_price": 50.0},
])

events = pd.DataFrame([
    {"user_id": "U1", "event_ts": "2026-01-01 10:00:00", "event_name": "login"},
    {"user_id": "U1", "event_ts": "2026-01-01 10:01:00", "event_name": "view"},
    {"user_id": "U1", "event_ts": "2026-01-01 10:03:00", "event_name": "logout"},
    {"user_id": "U2", "event_ts": "2026-01-01 11:00:00", "event_name": "login"},
    {"user_id": "U2", "event_ts": "2026-01-01 11:45:00", "event_name": "view"},
    {"user_id": "U2", "event_ts": "2026-01-01 12:10:00", "event_name": "purchase"},
    {"user_id": "U2", "event_ts": "2026-01-01 12:50:00", "event_name": "logout"},
    {"user_id": "U3", "event_ts": "2026-01-02 08:00:00", "event_name": "login"},
    {"user_id": "U3", "event_ts": "2026-01-02 08:20:00", "event_name": "view"},
])
events["event_ts"] = pd.to_datetime(events["event_ts"])

inventory_snapshots = pd.DataFrame([
    {"snapshot_date": "2026-01-01", "sku": "A", "quantity": 100},
    {"snapshot_date": "2026-01-02", "sku": "A", "quantity": 95},
    {"snapshot_date": "2026-01-03", "sku": "A", "quantity": 95},
    {"snapshot_date": "2026-01-01", "sku": "B", "quantity": 50},
    {"snapshot_date": "2026-01-03", "sku": "B", "quantity": 35},
    {"snapshot_date": "2026-01-01", "sku": "C", "quantity": 20},
    {"snapshot_date": "2026-01-02", "sku": "C", "quantity": 18},
    {"snapshot_date": "2026-01-03", "sku": "C", "quantity": 10},
])
inventory_snapshots["snapshot_date"] = pd.to_datetime(inventory_snapshots["snapshot_date"])

print(orders)
print(order_items)
print(events)
print(inventory_snapshots)

   order_id customer_id            order_ts  amount    status
0         1          C1 2026-01-01 10:00:00   120.0      paid
1         2          C2 2026-01-01 10:05:00    80.0      paid
2         3          C1 2026-01-01 11:00:00    50.0  refunded
3         4          C3 2026-01-02 09:15:00   200.0      paid
4         5          C2 2026-01-02 09:30:00    60.0      paid
5         6          C1 2026-01-03 12:00:00   150.0      paid
   order_id sku  qty  unit_price
0         1   A    2        40.0
1         1   B    1        40.0
2         2   B    2        40.0
3         3   C    1        50.0
4         4   A    1        40.0
5         4   D    4        40.0
6         5   C    1        60.0
7         6   A    3        50.0
  user_id            event_ts event_name
0      U1 2026-01-01 10:00:00      login
1      U1 2026-01-01 10:01:00       view
2      U1 2026-01-01 10:03:00     logout
3      U2 2026-01-01 11:00:00      login
4      U2 2026-01-01 11:45:00       view
5      U2 2026-01-01 12

## Medium 1: Customer Revenue Summary

Write a function `customer_revenue_summary(orders_df)` that returns one row per customer with:
- `customer_id`
- `total_paid_amount` counting only rows with `status == 'paid'`
- `paid_order_count`
- `avg_paid_order_amount`

Sort by `total_paid_amount` descending.

In [18]:
def customer_revenue_summary(orders):
    print(orders)
    paid_orders = orders[orders['status'] == 'paid']
    print(paid_orders)
    orders_summ = paid_orders.groupby('customer_id').agg(total_paid_amount=('amount','sum'),paid_order_count=('customer_id','count'),avg_paid_order_amount=('amount','mean')).reset_index()
    print(orders_summ.sort_values('total_paid_amount',ascending=False))
    return orders_summ.sort_values('total_paid_amount',ascending=False)


customer_revenue_summary(orders)

   order_id customer_id            order_ts  amount    status
0         1          C1 2026-01-01 10:00:00   120.0      paid
1         2          C2 2026-01-01 10:05:00    80.0      paid
2         3          C1 2026-01-01 11:00:00    50.0  refunded
3         4          C3 2026-01-02 09:15:00   200.0      paid
4         5          C2 2026-01-02 09:30:00    60.0      paid
5         6          C1 2026-01-03 12:00:00   150.0      paid
   order_id customer_id            order_ts  amount status
0         1          C1 2026-01-01 10:00:00   120.0   paid
1         2          C2 2026-01-01 10:05:00    80.0   paid
3         4          C3 2026-01-02 09:15:00   200.0   paid
4         5          C2 2026-01-02 09:30:00    60.0   paid
5         6          C1 2026-01-03 12:00:00   150.0   paid
  customer_id  total_paid_amount  paid_order_count  avg_paid_order_amount
0          C1              270.0                 2                  135.0
2          C3              200.0                 1              

,customer_id,total_paid_amount,paid_order_count,avg_paid_order_amount
0,C1,270.0,2,135.0
2,C3,200.0,1,200.0
1,C2,140.0,2,70.0


## Medium 2: Top SKU by Revenue

Join `orders` and `order_items` and compute revenue by `sku` using only paid orders.

Return a DataFrame with:
- `sku`
- `revenue`
- `units_sold`

Sort by revenue descending.

In [25]:

def top_sku_by_revenue(orders, items):
    item_orders = pd.merge(orders,items,on='order_id',how='inner')
    paid_orders = item_orders[item_orders['status'] == 'paid']
    print(paid_orders)
    agg_df = paid_orders.groupby('sku').agg(revenue=('amount','sum'),units_sold=('qty','sum')).reset_index()
    #print(item_orders)
    return agg_df.sort_values('revenue',ascending=False)

top_sku_by_revenue(orders,order_items)

   order_id customer_id            order_ts  amount status sku  qty  unit_price
0         1          C1 2026-01-01 10:00:00   120.0   paid   A    2        40.0
1         1          C1 2026-01-01 10:00:00   120.0   paid   B    1        40.0
2         2          C2 2026-01-01 10:05:00    80.0   paid   B    2        40.0
4         4          C3 2026-01-02 09:15:00   200.0   paid   A    1        40.0
5         4          C3 2026-01-02 09:15:00   200.0   paid   D    4        40.0
6         5          C2 2026-01-02 09:30:00    60.0   paid   C    1        60.0
7         6          C1 2026-01-03 12:00:00   150.0   paid   A    3        50.0


,sku,revenue,units_sold
0,A,470.0,6
1,B,200.0,3
3,D,200.0,4
2,C,60.0,1


## Medium 3: Sessionization with pandas

A new session starts if the gap between consecutive events for the same user is greater than 30 minutes.

Return a DataFrame with:
- `user_id`
- `session_id`
- `session_start`
- `session_end`
- `event_count`

Use pandas vectorized logic rather than Python loops.

In [ ]:
def build_sessions(events_df: pd.DataFrame, timeout_minutes: int = 30) -> pd.DataFrame:
    # TODO: implement
    raise NotImplementedError

build_sessions(events,timeout)

## Medium 4: Rolling 2-Day Customer Spend

For paid orders only, compute each customer's rolling 2-day spend based on `order_ts`.

Return the original paid orders plus a `rolling_2d_spend` column.

In [ ]:
def rolling_customer_spend(orders_df: pd.DataFrame) -> pd.DataFrame:
    # TODO: implement
    raise NotImplementedError

## Medium 5: Deduplicate by Latest Record

Given the DataFrame below, keep only the latest record per `id` based on `updated_at`.

If timestamps tie, keep the row with the largest `version`.

In [ ]:
records = pd.DataFrame([
    {"id": 1, "value": "A", "updated_at": "2026-01-01 10:00:00", "version": 1},
    {"id": 1, "value": "B", "updated_at": "2026-01-01 10:00:00", "version": 2},
    {"id": 2, "value": "X", "updated_at": "2026-01-03 09:00:00", "version": 1},
    {"id": 2, "value": "Y", "updated_at": "2026-01-02 09:00:00", "version": 2},
])
records["updated_at"] = pd.to_datetime(records["updated_at"])

def latest_records(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: implement
    raise NotImplementedError

## Hard 1: Incremental Merge Without SQL

Implement a Python function that simulates an upsert from `incoming_df` into `current_df`.

Rules:
- Match on `business_key`
- If incoming row is newer by `updated_at`, replace current row
- If key does not exist, insert it
- Keep unchanged rows

Return the merged DataFrame.

In [ ]:
current_df = pd.DataFrame([
    {"business_key": "K1", "value": 10, "updated_at": "2026-01-01 00:00:00"},
    {"business_key": "K2", "value": 20, "updated_at": "2026-01-02 00:00:00"},
])
incoming_df = pd.DataFrame([
    {"business_key": "K2", "value": 25, "updated_at": "2026-01-03 00:00:00"},
    {"business_key": "K3", "value": 30, "updated_at": "2026-01-01 12:00:00"},
])
current_df["updated_at"] = pd.to_datetime(current_df["updated_at"])
incoming_df["updated_at"] = pd.to_datetime(incoming_df["updated_at"])

def incremental_merge(current_df: pd.DataFrame, incoming_df: pd.DataFrame) -> pd.DataFrame:
    # TODO: implement
    raise NotImplementedError

## Hard 2: Detect Missing Inventory Dates per SKU

For each `sku`, generate the full date range from its min snapshot date to max snapshot date.
Return the missing dates per `sku`.

Expected columns:
- `sku`
- `missing_date`

In [ ]:
def missing_inventory_dates(snapshots_df: pd.DataFrame) -> pd.DataFrame:
    # TODO: implement
    raise NotImplementedError

## Hard 3: Build an Idempotent File Processor

Write a Python function `process_files(file_events)` where each event is a dict with:
- `file_name`
- `checksum`
- `received_at`

Requirements:
- Process a file only once per checksum
- If the same file arrives again with the same checksum, skip it
- If the same file arrives with a different checksum, process it again
- Return two lists: `processed`, `skipped`

Use a Python-only implementation.

In [ ]:
file_events = [
    {"file_name": "orders_20260101.csv", "checksum": "abc", "received_at": "2026-01-01T10:00:00"},
    {"file_name": "orders_20260101.csv", "checksum": "abc", "received_at": "2026-01-01T10:05:00"},
    {"file_name": "orders_20260101.csv", "checksum": "def", "received_at": "2026-01-01T10:10:00"},
    {"file_name": "customers_20260101.csv", "checksum": "xyz", "received_at": "2026-01-01T10:15:00"},
]

def process_files(file_events):
    # TODO: implement
    raise NotImplementedError

## Hard 4: Streaming Top K Keys

Implement `streaming_top_k(records, k)` that consumes an iterable of keys and returns the top `k` most frequent keys.

Questions to think about:
- Exact vs approximate solutions
- Memory usage
- How would this change for unbounded streams?

In [ ]:
def streaming_top_k(records, k):
    # TODO: implement
    raise NotImplementedError

## Hard 5: pandas Window Ranking per Day

For paid orders only, rank customers per calendar day by total daily spend.

Return columns:
- `order_date`
- `customer_id`
- `daily_spend`
- `spend_rank`

Use dense ranking. Highest spend gets rank 1.

In [ ]:
def daily_customer_rank(orders_df: pd.DataFrame) -> pd.DataFrame:
    # TODO: implement
    raise NotImplementedError

## Hard 6: Data Quality Checks Framework

Design and partially implement a mini data quality framework.

Create a function `run_checks(df, checks)` where `checks` is a list of callables.

Each check should return:
```python
{
    "check_name": str,
    "passed": bool,
    "failed_rows": int,
    "message": str,
}
```

Implement at least these checks:
- no nulls in required columns
- uniqueness of a key column
- amount column must be non-negative

In [ ]:
def run_checks(df: pd.DataFrame, checks):
    # TODO: implement
    raise NotImplementedError

## Validation Sandbox

Use this section to run your own checks after implementing solutions.

In [ ]:
# Example manual runs after implementation:
# customer_revenue_summary(orders)
# top_sku_by_revenue(orders, order_items)
# build_sessions(events)
# rolling_customer_spend(orders)
# latest_records(records)
# incremental_merge(current_df, incoming_df)
# missing_inventory_dates(inventory_snapshots)
# process_files(file_events)
# streaming_top_k(["a", "b", "a", "c", "a", "b"], 2)
# daily_customer_rank(orders)